In [0]:
-- Volume & Ingestion Audit
SELECT 'sales_customers' AS table_name, COUNT(*) AS row_count FROM olist_maas_pipeline.bronze.sales_customers
UNION ALL
SELECT 'sales_geolocation', COUNT(*) FROM olist_maas_pipeline.bronze.sales_geolocation
UNION ALL
SELECT 'sales_order_items', COUNT(*) FROM olist_maas_pipeline.bronze.sales_order_items
UNION ALL
SELECT 'sales_order_payments', COUNT(*) FROM olist_maas_pipeline.bronze.sales_order_payments
UNION ALL
SELECT 'sales_order_reviews', COUNT(*) FROM olist_maas_pipeline.bronze.sales_order_reviews
UNION ALL
SELECT 'sales_orders', COUNT(*) FROM olist_maas_pipeline.bronze.sales_orders
UNION ALL
SELECT 'sales_products', COUNT(*) FROM olist_maas_pipeline.bronze.sales_products
UNION ALL
SELECT 'sales_sellers', COUNT(*) FROM olist_maas_pipeline.bronze.sales_sellers
UNION ALL
SELECT 'sales_product_category_translation', COUNT(*) FROM olist_maas_pipeline.bronze.sales_product_category_translation
UNION ALL
SELECT 'marketing_closed_deals', COUNT(*) FROM olist_maas_pipeline.bronze.marketing_closed_deals
UNION ALL
SELECT 'marketing_mql', COUNT(*) FROM olist_maas_pipeline.bronze.marketing_mql
UNION ALL
SELECT 'testing_marketing_ab', COUNT(*) FROM olist_maas_pipeline.bronze.testing_marketing_ab
ORDER BY table_name;

In [0]:
-- Uniqueness Check
-- 1. Fact Tables 
SELECT 'sales_orders' AS tbl, 'order_id' AS pk, (COUNT(order_id) - COUNT(DISTINCT order_id)) AS dups FROM olist_maas_pipeline.bronze.sales_orders
UNION ALL
SELECT 'sales_order_payments', 'order_id+payment_sequential', (COUNT(*) - COUNT(DISTINCT order_id, payment_sequential)) FROM  olist_maas_pipeline.bronze.sales_order_payments
UNION ALL
SELECT 'sales_order_items', 'order_id+item_id', (COUNT(*) - COUNT(DISTINCT order_id, order_item_id)) FROM  olist_maas_pipeline.bronze.sales_order_items
UNION ALL
SELECT 'marketing_closed_deals', 'mql_id', (COUNT(mql_id) - COUNT(DISTINCT mql_id)) FROM  olist_maas_pipeline.bronze.marketing_closed_deals
UNION ALL
SELECT 'marketing_mql', 'mql_id', (COUNT(mql_id) - COUNT(DISTINCT mql_id)) FROM  olist_maas_pipeline.bronze.marketing_mql

-- 2. Dimension Tables (for join)
UNION ALL
SELECT 'sales_customers', 'customer_id', (COUNT(customer_id) - COUNT(DISTINCT customer_id)) FROM  olist_maas_pipeline.bronze.sales_customers
UNION ALL
SELECT 'sales_products', 'product_id', (COUNT(product_id) - COUNT(DISTINCT product_id)) FROM  olist_maas_pipeline.bronze.sales_products
UNION ALL
SELECT 'sales_sellers', 'seller_id', (COUNT(seller_id) - COUNT(DISTINCT seller_id)) FROM  olist_maas_pipeline.bronze.sales_sellers;


In [0]:
-- Null Profiling
-- Revenue standpoint
SELECT 
  'sales_order_items' AS table_name,
  'price' AS column_name,
  COUNT(*) AS total_rows,
  SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS null_count,
  ROUND(AVG(CASE WHEN price IS NULL THEN 1 ELSE 0 END) * 100, 2) AS null_pct
FROM  olist_maas_pipeline.bronze.sales_order_items

UNION ALL

--  Operation standpoint
SELECT 
  'sales_orders',
  'order_purchase_timestamp',
  COUNT(*),
  SUM(CASE WHEN order_purchase_timestamp IS NULL THEN 1 ELSE 0 END),
  ROUND(AVG(CASE WHEN order_purchase_timestamp IS NULL THEN 1 ELSE 0 END) * 100, 2)
FROM  olist_maas_pipeline.bronze.sales_orders

UNION ALL

-- Acquisition standpoint
SELECT 
  'marketing_mql',
  'origin',
  COUNT(*),
  SUM(CASE WHEN origin IS NULL THEN 1 ELSE 0 END),
  ROUND(AVG(CASE WHEN origin IS NULL THEN 1 ELSE 0 END) * 100, 2)
FROM  olist_maas_pipeline.bronze.marketing_mql;

-- Referential integrity: Look for FK that points to nothing, e.g., items sold without connecting to a seller
SELECT 
  'Order Items -> Sellers' AS path,
  COUNT(i.order_id) AS orphaned_rows
FROM  olist_maas_pipeline.bronze.sales_order_items i
LEFT JOIN  olist_maas_pipeline.bronze.sales_sellers s ON i.seller_id = s.seller_id
WHERE s.seller_id IS NULL;


In [0]:
-- Numerical Range & Logic Audit
SELECT 
  'sales_order_items' AS table_name,
  'price' AS column_name,
  MIN(price) AS min_val,
  MAX(price) AS max_val,
  SUM(CASE WHEN price <= 0 THEN 1 ELSE 0 END) AS invalid_records
FROM  olist_maas_pipeline.bronze.sales_order_items

UNION ALL

-- Checking the "Breaking Point" Logic
-- Shipping should not be free unless specified; very high freight is also a red flag.
SELECT 
  'sales_order_items',
  'freight_value',
  MIN(freight_value),
  MAX(freight_value),
  SUM(CASE WHEN freight_value < 0 THEN 1 ELSE 0 END)
FROM  olist_maas_pipeline.bronze.sales_order_items

UNION ALL

-- Review Score Sanity
SELECT 
  'sales_order_reviews',
  'review_score',
  MIN(review_score),
  MAX(review_score),
  SUM(CASE WHEN review_score < 1 OR review_score > 5 THEN 1 ELSE 0 END)
FROM  olist_maas_pipeline.bronze.sales_order_reviews;

-- Date Check
SELECT 
  MIN(order_purchase_timestamp) AS earliest_order,
  MAX(order_purchase_timestamp) AS latest_order,
  COUNT(*) FILTER (WHERE order_purchase_timestamp > '2019-01-01') AS post_dataset_records
FROM  olist_maas_pipeline.bronze.sales_orders;

-- Temporal Logic Audit
-- Goal: All counts should be 0.
SELECT 
  'Delivery before Purchase' AS logic_error,
  COUNT(*) AS error_count
FROM  olist_maas_pipeline.bronze.sales_orders
WHERE order_delivered_customer_date < order_purchase_timestamp

UNION ALL

-- Checking for "Time Travelers" (Orders in the future)
SELECT 
  'Future Dated Orders',
  COUNT(*)
FROM  olist_maas_pipeline.bronze.sales_orders
WHERE order_purchase_timestamp > current_date();